# 07 — LLM Extraction with Reflection (Milestone M3)

**DSML stage:** modeling. Turns chunks (notebook 04) into **candidate graph knowledge**:
risk factors, products, and relation triples (`SUPPLIES_TO`, `DEPENDS_ON`, `CUSTOMER_OF`, `COMPETES_WITH`).

Anti-hallucination design (per the feasibility studies):
1. **Schema-constrained extraction** — the LLM must emit JSON matching strict Pydantic models
2. **Verbatim evidence quotes** — every claim carries a quote; a *programmatic* substring check kills
   fabricated evidence before any LLM judges it
3. **Critic reflection** — a second LLM pass re-reads the chunk and discards unsupported triples
4. **Persist before loading** — outputs go to `data/processed/extractions/*.jsonl` (checkpointed,
   resumable); the graph load happens separately (notebook 09)

Provider-agnostic via **LiteLLM**; default model is Claude Sonnet (`LLM_MODEL` in `.env`).

In [1]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError
from litellm import completion

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

LLM_MODEL = os.environ["LLM_MODEL"]
# --- prerequisite guard ---
if LLM_MODEL.startswith("anthropic/") and not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY is empty in .env — add your key (console.anthropic.com) to run extraction.")

EXTRACTIONS_DIR = PROJECT_ROOT / "data" / "processed" / "extractions"
EXTRACTIONS_DIR.mkdir(parents=True, exist_ok=True)
PROMPTS_DIR = PROJECT_ROOT / "artifacts" / "prompts"
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

chunks_df = pd.read_parquet(PROJECT_ROOT / "data/processed/chunks/nvda_chunks.parquet")
print(f"{len(chunks_df)} chunks available | model: {LLM_MODEL}")

439 chunks available | model: anthropic/claude-sonnet-5


## 1. Output schemas (Pydantic) — the contract the LLM must satisfy

In [2]:
RELATION_TYPES = ["SUPPLIES_TO", "DEPENDS_ON", "CUSTOMER_OF", "COMPETES_WITH"]
RISK_CATEGORIES = ["Supply Chain", "Geopolitical", "Export Controls", "Demand", "Competition",
                   "Technology", "Legal/Regulatory", "Financial", "Other"]

class Relation(BaseModel):
    source_entity: str = Field(description="Organization/product doing the acting, exactly as named in the text")
    relation: str = Field(description=f"One of {RELATION_TYPES}")
    target_entity: str = Field(description="The other organization/product, exactly as named in the text")
    evidence_quote: str = Field(description="VERBATIM quote (<=40 words) from the chunk that states this relationship")

class RiskFactor(BaseModel):
    summary: str = Field(description="One-sentence summary of the specific risk")
    category: str = Field(description=f"One of {RISK_CATEGORIES}")
    evidence_quote: str = Field(description="VERBATIM quote (<=40 words) from the chunk")

class Product(BaseModel):
    name: str = Field(description="Product/technology name as written, e.g. 'Blackwell', 'HBM3e', 'CoWoS'")
    type: str = Field(description="GPU | CPU | ASIC | Memory | Packaging | Equipment | Platform | Other")

class ChunkExtraction(BaseModel):
    relations: list[Relation] = []
    risk_factors: list[RiskFactor] = []
    products: list[Product] = []

class CriticVerdict(BaseModel):
    verdicts: list[bool] = Field(description="For each numbered claim, true only if the chunk explicitly supports it")

## 2. Prompts (versioned artifacts) and LLM helpers

In [ ]:
EXTRACTOR_PROMPT = """You extract structured facts from SEC filing excerpts for a semiconductor supply-chain knowledge graph.

Filing context: {ticker} {form} filed {filing_date}, section \"{section_title}\"{sub_heading}.

Extract ONLY what the text explicitly states:
1. relations: supplier/customer/dependency/competition links between NAMED organizations or products.
   Allowed types: SUPPLIES_TO (A supplies goods/services to B), DEPENDS_ON (A relies on B),
   CUSTOMER_OF (A buys from B), COMPETES_WITH (A competes with B).
   The filer refers to itself as \"we\"/\"our\" — resolve that to {ticker_name}.
   NEVER infer a relation that is not stated. Generic mentions (\"our suppliers\") without a name are NOT relations.
2. risk_factors: specific disclosed risks (only if this is a risk-factors section or the text plainly discloses a risk).
3. products: named products/technologies (chip families, memory types, packaging tech).

Every relation and risk_factor MUST include evidence_quote: an exact, verbatim quote from the text (<=40 words).
If nothing qualifies, return empty lists.

Respond with ONLY a JSON object matching this schema (no markdown fences):
{schema}

TEXT:
{chunk_text}"""

CRITIC_PROMPT = """You are a strict fact-checker. Below is an excerpt from a SEC filing, followed by numbered claims
extracted from it. For each claim, answer true ONLY if the excerpt explicitly supports it — no outside knowledge,
no inference beyond the text. Answer false if the claim overstates, reverses direction, or is not in the excerpt.

EXCERPT:
{chunk_text}

CLAIMS:
{claims}

Respond with ONLY a JSON object: {{\"verdicts\": [true/false, ...]}} — one verdict per claim, in order."""

(PROMPTS_DIR / "extractor.txt").write_text(EXTRACTOR_PROMPT, encoding="utf-8")
(PROMPTS_DIR / "critic.txt").write_text(CRITIC_PROMPT, encoding="utf-8")

def llm_json(prompt: str, model_cls, max_tokens: int = 2000):
    """Call the LLM, parse+validate JSON against a Pydantic model; one retry with error feedback.
    Note: no temperature/top_p — Claude Sonnet 5 rejects non-default sampling params (400)."""
    messages = [{"role": "user", "content": prompt}]
    for attempt in range(2):
        resp = completion(model=LLM_MODEL, messages=messages, max_tokens=max_tokens)
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()
        try:
            return model_cls.model_validate_json(raw)
        except ValidationError as e:
            if attempt == 1:
                raise
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": f"Invalid JSON for the schema: {e}. Reply with corrected JSON only."}]

def normalize(s: str) -> str:
    return re.sub(r"[\s’'\"“”]+", " ", s).strip().lower()

def quote_in_chunk(quote: str, chunk_text: str) -> bool:
    """Programmatic anti-fabrication gate: the evidence quote must actually appear in the chunk."""
    return normalize(quote) in normalize(chunk_text)

## 3. Scope & cost estimate

PoC scope: the **latest 10-K** (Business, Risk Factors, MD&A) + the **latest 10-Q** (MD&A, Risk Factors).
Historical 10-Ks join in Phase 5 when temporal versioning (notebook 13) can consume them.

In [4]:
latest_10k = chunks_df[chunks_df["form"] == "10-K"]["accession_no"].max()
latest_10q = chunks_df[chunks_df["form"] == "10-Q"]["accession_no"].max()
scope = chunks_df[chunks_df["accession_no"].isin([latest_10k, latest_10q])].reset_index(drop=True)

in_tokens = scope["n_tokens"].sum()
est = (in_tokens * 2 / 1e6 * 3) + (len(scope) * 600 / 1e6 * 15)  # 2 passes in @$3/M, ~600 out-tokens @$15/M
print(f"Scope: {len(scope)} chunks, {in_tokens:,} input tokens (extractor + critic passes)")
print(f"Estimated cost: ~${est:.2f}")
scope.groupby(["form", "section_id"]).size()

Scope: 130 chunks, 48,218 input tokens (extractor + critic passes)
Estimated cost: ~$1.46


form  section_id
10-K  I.1           21
      I.1A          36
      II.7          33
10-Q  I.2           31
      II.1A          9
dtype: int64

## 4. Run — checkpointed and resumable (safe to interrupt and re-run)

In [5]:
schema_json = json.dumps(ChunkExtraction.model_json_schema(), indent=None)
out_path = EXTRACTIONS_DIR / "nvda_extractions.jsonl"
done = set()
if out_path.exists():
    done = {json.loads(line)["chunk_id"] for line in out_path.open(encoding="utf-8")}
print(f"{len(done)} chunks already extracted — resuming")

with out_path.open("a", encoding="utf-8") as sink:
    for i, c in scope.iterrows():
        if c["chunk_id"] in done:
            continue
        prompt = EXTRACTOR_PROMPT.format(
            ticker=c["ticker"], ticker_name="Nvidia", form=c["form"], filing_date=c["filing_date"],
            section_title=c["section_title"],
            sub_heading=f", sub-heading \"{c['sub_heading']}\"" if c["sub_heading"] else "",
            schema=schema_json, chunk_text=c["text"],
        )
        extraction = llm_json(prompt, ChunkExtraction)

        # Gate 1 (programmatic): evidence quotes must exist verbatim in the chunk
        relations = [r for r in extraction.relations if quote_in_chunk(r.evidence_quote, c["text"])]
        risks = [r for r in extraction.risk_factors if quote_in_chunk(r.evidence_quote, c["text"])]
        n_dropped_quotes = (len(extraction.relations) - len(relations)) + (len(extraction.risk_factors) - len(risks))

        # Gate 2 (critic reflection): semantic support check on surviving relations
        if relations:
            claims = "\n".join(
                f"{j+1}. {r.source_entity} {r.relation} {r.target_entity}" for j, r in enumerate(relations)
            )
            verdict = llm_json(CRITIC_PROMPT.format(chunk_text=c["text"], claims=claims), CriticVerdict, max_tokens=300)
            kept = [r for r, ok in zip(relations, verdict.verdicts) if ok] if len(verdict.verdicts) == len(relations) else relations
        else:
            kept = []

        record = {
            "chunk_id": c["chunk_id"], "accession_no": c["accession_no"], "section_id": c["section_id"],
            "relations": [r.model_dump() for r in kept],
            "risk_factors": [r.model_dump() for r in risks],
            "products": [p.model_dump() for p in extraction.products],
            "dropped_bad_quotes": n_dropped_quotes,
            "dropped_by_critic": len(relations) - len(kept),
        }
        sink.write(json.dumps(record) + "\n")
        sink.flush()
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(scope)} chunks")
print("Extraction pass complete")

0 chunks already extracted — resuming


UnsupportedParamsError: litellm.UnsupportedParamsError: claude-sonnet-5 does not support temperature=0.0. Only temperature=1 is supported. To drop unsupported params, set `litellm.drop_params = True`.

## 5. Review the harvest

In [ ]:
records = [json.loads(line) for line in out_path.open(encoding="utf-8")]
rel_rows = [dict(r, chunk_id=rec["chunk_id"]) for rec in records for r in rec["relations"]]
rels_df = pd.DataFrame(rel_rows)
print(f"{len(records)} chunks → {len(rels_df)} relations, "
      f"{sum(len(r['risk_factors']) for r in records)} risk factors, "
      f"{sum(len(r['products']) for r in records)} product mentions")
print(f"Dropped: {sum(r['dropped_bad_quotes'] for r in records)} fabricated-quote, "
      f"{sum(r['dropped_by_critic'] for r in records)} critic-rejected")
rels_df.groupby("relation").size().sort_values(ascending=False) if len(rels_df) else rels_df

In [ ]:
# Manual verification sample — read these against the quotes (M3 verification step)
if len(rels_df):
    with pd.option_context("display.max_colwidth", 120):
        display(rels_df.sample(min(10, len(rels_df)), random_state=7)[
            ["source_entity", "relation", "target_entity", "evidence_quote"]])

In [ ]:
# --- M3 (extraction) assertion cell ---
assert len(records) == len(scope), f"extraction incomplete: {len(records)}/{len(scope)} chunks"
assert len(rels_df) >= 10, "suspiciously few relations from an NVDA 10-K — inspect prompts"
chunk_text_by_id = dict(zip(chunks_df["chunk_id"], chunks_df["text"]))
for rec in records:
    for r in rec["relations"] + rec["risk_factors"]:
        assert quote_in_chunk(r["evidence_quote"], chunk_text_by_id[rec["chunk_id"]]), \
            f"unverifiable quote persisted in {rec['chunk_id']}"
assert rels_df["relation"].isin(RELATION_TYPES).all()
tsmc_rels = rels_df[rels_df.apply(lambda r: "tsmc" in (r["source_entity"] + r["target_entity"]).lower(), axis=1)]
assert len(tsmc_rels) > 0, "expected at least one TSMC relation from NVDA filings"
print(f"M3 (extraction) OK — {len(rels_df)} verified relations (incl. {len(tsmc_rels)} touching TSMC), all quotes verbatim")